# Reproducing the DoubleML / hdm 401(k) result with StatsPAI

The effect of **401(k) eligibility on net financial assets** is the canonical
applied example for double machine learning (DoubleML's "Getting Started"
vignette; Chernozhukov & Hansen 2004; the `hdm` package). This notebook shows
that `sp.dml` reproduces that result, **side by side with `doubleml-for-py`**
on the *same* data.

Unlike the offline scripts in `examples/`, this notebook is **networked**: it
fetches the data from DoubleML's own public distribution, so StatsPAI ships no
copy of it. Requirements:

```bash
pip install statspai doubleml scikit-learn
```

See `docs/guides/case_study_401k.md` for the narrative version and
`docs/guides/sp_dml_vs_doubleml.md` for the bit-for-bit parity discussion.

In [1]:
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import statspai as sp
import doubleml as dml
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from doubleml.datasets import fetch_401K

# Canonical 401(k) data, fetched from DoubleML's public source (no bundling).
df = fetch_401K(return_type="DataFrame")
y_col, d_col = "net_tfa", "e401"
covariates = [c for c in df.columns if c not in (y_col, d_col, "p401")]
print(f"401(k): {df.shape[0]} households, treatment={d_col}, outcome={y_col}")
df.head()

401(k): 9915 households, treatment=e401, outcome=net_tfa


,nifa,net_tfa,tw,age,inc,fsize,educ,db,marr,twoearn,e401,p401,pira,hown
0,0.0,0.0,4500.0,47,6765.0,2,8,0,0,0,0,0,0,1
1,6215.0,1015.0,22390.0,36,28452.0,1,16,0,0,0,0,0,0,1
2,0.0,-2000.0,-2000.0,37,3300.0,6,12,1,0,0,0,0,0,0
3,15000.0,15000.0,155000.0,58,52590.0,2,16,0,1,1,0,0,0,1
4,0.0,0.0,58000.0,32,21804.0,1,11,0,0,0,0,0,0,1


## Estimate with `sp.dml`

Partially linear (PLR) and interactive/AIPW (IRM) double-ML, plus the
rigorous-Lasso (`hdm`) nuisance path. `random_state` makes each fully
reproducible.

In [2]:
plr = sp.dml(data=df, y=y_col, treat=d_col, covariates=covariates,
             model="plr", ml_g="lasso", ml_m="lasso", n_folds=5, random_state=42)
irm = sp.dml(data=df, y=y_col, treat=d_col, covariates=covariates,
             model="irm", ml_g="lasso", ml_m="logistic", n_folds=5, random_state=42)
plr_hdm = sp.dml(data=df, y=y_col, treat=d_col, covariates=covariates,
                 model="plr", ml_g="rlasso", ml_m="rlasso", n_folds=5, random_state=42)
print(f"StatsPAI PLR (lasso)          : {plr.estimate:8.1f}  (se {plr.se:.1f})")
print(f"StatsPAI IRM ATE (lasso+logit): {irm.estimate:8.1f}  (se {irm.se:.1f})")
print(f"StatsPAI PLR (rlasso/hdm)     : {plr_hdm.estimate:8.1f}  (se {plr_hdm.se:.1f})")

StatsPAI PLR (lasso)          :   8227.7  (se 566.3)
StatsPAI IRM ATE (lasso+logit):   8644.5  (se 1742.6)
StatsPAI PLR (rlasso/hdm)     :   8302.8  (se 469.3)


## Cross-check with `doubleml-for-py`

The same `DoubleMLPLR` / `DoubleMLIRM` on the identical `DataFrame`. We seed
NumPy before each fit so DoubleML's sample splitting is reproducible.

In [3]:
dd = dml.DoubleMLData(df, y_col=y_col, d_cols=d_col, x_cols=covariates)
np.random.seed(42)
m_plr = dml.DoubleMLPLR(dd, ml_l=LassoCV(), ml_m=LassoCV(), n_folds=5).fit()
np.random.seed(42)
m_irm = dml.DoubleMLIRM(dd, ml_g=LassoCV(), ml_m=LogisticRegressionCV(), n_folds=5).fit()

compare = pd.DataFrame(
    {
        "StatsPAI estimate": [plr.estimate, irm.estimate],
        "StatsPAI se": [plr.se, irm.se],
        "DoubleML estimate": [m_plr.coef[0], m_irm.coef[0]],
        "DoubleML se": [m_plr.se[0], m_irm.se[0]],
    },
    index=["PLR", "IRM (ATE)"],
).round(1)
compare

,StatsPAI estimate,StatsPAI se,DoubleML estimate,DoubleML se
PLR,8227.7,566.3,8227.7,566.3
IRM (ATE),8644.5,1742.6,9584.1,2723.3


## Reading the result

Both engines estimate a large positive effect of 401(k) eligibility on net
financial assets, in the **\$8,000–\$10,000** range reported throughout the
DoubleML and `hdm` literature.

The **partially linear (PLR)** rows are *identical to the displayed precision*
(see the table above): under the same seed and the same `LassoCV` learners,
`sp.dml` and `doubleml-for-py` evaluate the same Neyman-orthogonal score on the
same cross-fitting partition — the machine-precision PLR parity, live. The
**interactive/AIPW (IRM)** estimand is higher-variance: here the two engines
differ by roughly half a standard error (independent fold construction plus
propensity-score variability), both pointing to the same sizable positive
effect.

The residual IRM gap is **fold/propensity randomization**, not a method
discrepancy. Bit-for-bit PLR/PLIV parity under shared folds — and the full
divergence discussion — are pinned offline (no network) in
`tests/external_parity/test_dml_python_parity.py` and explained in
`docs/guides/sp_dml_vs_doubleml.md`.